In [1]:
import os
import kagglehub

# Download the dataset again and get the correct path
path = kagglehub.dataset_download("saurabhshahane/urdu-news-dataset")
print(f"Download path: {path}")

# List all files in the downloaded directory
print("Files in directory:")
for file in os.listdir(path):
    print(f"  - {file}")

# Check if any CSV files exist
csv_files = [f for f in os.listdir(path) if f.endswith('.csv')]
print(f"\nCSV files found: {csv_files}")

# Use the first CSV file found
if csv_files:
    file_path = os.path.join(path, csv_files[0])
    print(f"Using file: {file_path}")
else:
    print("No CSV files found in the directory")

Download path: /home/cvl/.cache/kagglehub/datasets/saurabhshahane/urdu-news-dataset/versions/1
Files in directory:
  - urdu-news-dataset-1M.csv

CSV files found: ['urdu-news-dataset-1M.csv']
Using file: /home/cvl/.cache/kagglehub/datasets/saurabhshahane/urdu-news-dataset/versions/1/urdu-news-dataset-1M.csv


In [2]:
!pip install python-docx

In [1]:
# =============================================================================
# ENHANCED DATA PREPROCESSING PIPELINE FOR URDU NEWS DATASET
# =============================================================================

import pandas as pd
import numpy as np
import re
import os
from pathlib import Path

class UrduNewsPreprocessor:
    """
    A comprehensive preprocessing pipeline for Urdu news text data
    designed for transformer-based content recommendation systems.
    """

    def __init__(self, file_path: str, output_dir: str = "processed_data"):
        self.file_path = file_path
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(exist_ok=True)
        self.df = None
        self.stopwords_removed_count = 0
        self.total_words_before = 0

        # Comprehensive Urdu stop words list
        self.urdu_stop_words = [
            "اور", "ہے", "ہیں", "کو", "سے", "میں", "پر", "نے", "تھا", "تھی",
            "تھے", "ہے۔", "کا۔", "کی", "کے", "کا", "ہی", "ہوا", "ہوئی", "ہوئے",
            "کر", "گی", "گا", "گے", "لیے", "والے", "والی", "والا", "رہی", "رہا",
            "رہے", "دی", "دیا", "دیں", "دیتا", "دیتی", "دیتے", "سکتا", "سکتی",
            "سکتے", "چاہیے", "چاہتا", "چاہتی", "چاہتے", "کیوں", "کہ", "جب", "تو",
            "پھر", "بھی", "یا", "اگر", "تو", "ورنہ", "لہذا", "کیونکہ", "جبکہ",
            "پہلے", "اب", "آج", "کل", "سے", "تک", "پر", "میں", "کے", "کے", "سے",
            "تک", "پر", "میں", "کے", "لیے", "کے", "ساتھ", "کے", "بغیر", "کے", "بارے",
            "کے", "خلاف", "کے", "مطابق", "کے", "علاوہ", "کے", "باہر", "کے", "اندر",
            "کے", "اوپر", "کے", "نیچے", "کے", "آگے", "کے", "پیچھے", "کے", "پاس",
            "کے", "قریب", "کے", "دور", "یہ", "وہ", "جو", "کون", "کس", "کسی", "کچھ",
            "سب", "کوئی", "ہر", "اکثر", "کبھی", "ہمیشہ", "شاید", "ضرور", "قطعاً",
            "بہت", "زیادہ", "کم", "تھوڑا", "صرف", "تقریباً", "بالکل", "فیصد"
        ]

    def load_and_decode_dataset(self):
        """Load the dataset with proper encoding handling for Urdu text."""
        print("Loading Urdu news dataset...")

        df = pd.read_csv(self.file_path, encoding="ISO-8859-1")
        print(f"Initial dataset dimensions: {df.shape}")
        print(f"Dataset columns: {df.columns.tolist()}")

        def correct_text_encoding(text):
            """Correct encoding issues in Urdu text data."""
            try:
                return text.encode("latin1").decode("utf-8")
            except (UnicodeEncodeError, UnicodeDecodeError):
                return text

        text_columns = ["Headline", "News Text", "Category", "Source"]
        for col in text_columns:
            if col in df.columns:
                df[col] = df[col].astype(str).apply(correct_text_encoding)
                print(f"Applied encoding correction to: {col}")

        return df

    def remove_irrelevant_features(self, df):
        """Remove metadata columns not required for text analysis."""
        columns_to_drop = ["Date", "URL", "News length"]
        df_clean = df.drop(columns=columns_to_drop, errors='ignore')

        dropped_cols = set(columns_to_drop) & set(df.columns)
        if dropped_cols:
            print(f"Removed irrelevant columns: {list(dropped_cols)}")

        return df_clean

    def create_composite_content(self, df):
        """Combine headline and news text to create comprehensive content."""
        df_processed = df.copy()

        headline = df.get("Headline", "")
        news_text = df.get("News Text", "")
        df_processed["content"] = headline + " " + news_text
        df_processed["content_before_cleaning"] = df_processed["content"]  # Store original for comparison
        print("Created composite content from headline and news text")

        return df_processed

    def clean_urdu_text(self, text):
        """Perform comprehensive cleaning of Urdu text data."""
        if not isinstance(text, str) or len(text.strip()) == 0:
            return ""

        # Remove HTML tags
        text = re.sub(r"<[^>]+>", " ", text)

        # Remove English characters and numbers (optional - can be modified)
        text = re.sub(r"[a-zA-Z0-9]", " ", text)

        # Remove punctuation (both Urdu and English)
        urdu_english_punctuation = r"[!\"#$%&'()*+,\-./:;<=>?@\[\\\]^_`{|}~۔،؛؟٪]"
        text = re.sub(urdu_english_punctuation, " ", text)

        # Normalize whitespace
        text = re.sub(r"\s+", " ", text)

        return text.strip()

    def remove_urdu_stopwords(self, text):
        """Remove Urdu stop words from text while preserving semantics."""
        if not isinstance(text, str) or len(text.strip()) == 0:
            return ""

        # Split text into words
        words = text.split()

        # Count words before stop word removal for statistics
        words_before = len(words)

        # Filter out stop words
        filtered_words = [word for word in words if word not in self.urdu_stop_words]

        # Count stop words removed
        stopwords_removed = words_before - len(filtered_words)
        self.stopwords_removed_count += stopwords_removed
        self.total_words_before += words_before

        # Join back into text
        cleaned_text = ' '.join(filtered_words)

        return cleaned_text.strip()

    def remove_invalid_entries(self, df):
        """Remove null values, duplicates, and short texts."""
        initial_count = len(df)

        df = df.dropna(subset=["content", "Category"])
        after_null_removal = len(df)

        df = df.drop_duplicates(subset=["content"])
        after_deduplication = len(df)

        # Increased minimum length after stop word removal
        df = df[df["content"].str.len() > 15]
        final_count = len(df)

        df = df.reset_index(drop=True)

        print("Data cleaning summary:")
        print(f"  - Removed {initial_count - after_null_removal} null entries")
        print(f"  - Removed {after_null_removal - after_deduplication} duplicate entries")
        print(f"  - Removed {after_deduplication - final_count} short texts")
        print(f"  - Final dataset size: {final_count} records")

        return df

    def execute_preprocessing_pipeline(self):
        """Execute the complete preprocessing pipeline."""
        print("Starting Urdu news dataset preprocessing pipeline")
        print("=" * 60)

        # Reset counters
        self.stopwords_removed_count = 0
        self.total_words_before = 0

        # Step 1: Load and decode dataset
        df = self.load_and_decode_dataset()

        # Step 2: Remove irrelevant features
        df = self.remove_irrelevant_features(df)

        # Step 3: Create composite content
        df = self.create_composite_content(df)

        # Step 4: Clean Urdu text (basic cleaning)
        print("Applying basic text cleaning...")
        df["content"] = df["content"].apply(self.clean_urdu_text)

        # Step 5: Remove Urdu stop words
        print("Removing Urdu stop words...")
        df["content"] = df["content"].apply(self.remove_urdu_stopwords)

        # Step 6: Remove invalid entries
        df = self.remove_invalid_entries(df)

        self.df = df

        # Save processed dataset
        output_path = self.output_dir / "cleaned_urdu_news.csv"
        df.to_csv(output_path, index=False, encoding='utf-8')
        print(f"Processed dataset saved to: {output_path}")

        # Generate preprocessing statistics
        self.generate_preprocessing_stats(df)

        return df

    def generate_preprocessing_stats(self, df):
        """Generate statistics about the preprocessing results."""
        print("\n" + "="*50)
        print("PREPROCESSING STATISTICS")
        print("="*50)

        # Calculate average content length
        content_lengths = df["content"].str.len()
        avg_length = content_lengths.mean()

        # Calculate actual stop words removal percentage
        stopwords_removed_percentage = (self.stopwords_removed_count / self.total_words_before * 100) if self.total_words_before > 0 else 0

        print(f"Final dataset shape: {df.shape}")
        print(f"Average content length: {avg_length:.1f} characters")
        print(f"Total words processed: {self.total_words_before:,}")
        print(f"Stop words removed: {self.stopwords_removed_count:,}")
        print(f"Stop words removal rate: {stopwords_removed_percentage:.1f}%")
        print(f"Categories in dataset: {df['Category'].nunique()}")
        print(f"Sources in dataset: {df['Source'].nunique()}")

        # Show before/after comparison for a few samples
        print("\nStop Word Removal Examples:")
        print("-" * 40)
        sample_df = df.head(3)
        for idx, row in sample_df.iterrows():
            original_text = row.get('content_before_cleaning', 'N/A')[:100] + "..." if isinstance(row.get('content_before_cleaning'), str) else "N/A"
            cleaned_text = row['content'][:100] + "..." if len(row['content']) > 100 else row['content']

            print(f"\nSample {idx + 1}:")
            print(f"Before: {original_text}")
            print(f"After:  {cleaned_text}")
            print(f"Length reduced: {len(original_text) - len(row['content']) if isinstance(original_text, str) else 0} characters")

# =============================================================================
# MAIN EXECUTION
# =============================================================================

# Initialize preprocessor
file_path = "/home/cvl/.cache/kagglehub/datasets/saurabhshahane/urdu-news-dataset/versions/1/urdu-news-dataset-1M.csv"
preprocessor = UrduNewsPreprocessor(file_path)

# Execute preprocessing pipeline
df_cleaned = preprocessor.execute_preprocessing_pipeline()

# Display final results
print("\n" + "="*60)
print("PREPROCESSING COMPLETED SUCCESSFULLY")
print("="*60)
print(f"Final dataset shape: {df_cleaned.shape}")
print(f"Columns in final dataset: {df_cleaned.columns.tolist()}")

Starting Urdu news dataset preprocessing pipeline
Loading Urdu news dataset...
Initial dataset dimensions: (111861, 8)
Dataset columns: ['ï»¿Index', 'Headline', 'News Text', 'Category', 'Date', 'URL', 'Source', 'News length']
Applied encoding correction to: Headline
Applied encoding correction to: News Text
Applied encoding correction to: Category
Applied encoding correction to: Source
Removed irrelevant columns: ['Date', 'URL', 'News length']
Created composite content from headline and news text
Applying basic text cleaning...
Removing Urdu stop words...
Data cleaning summary:
  - Removed 0 null entries
  - Removed 8 duplicate entries
  - Removed 0 short texts
  - Final dataset size: 111853 records
Processed dataset saved to: processed_data/cleaned_urdu_news.csv

PREPROCESSING STATISTICS
Final dataset shape: (111853, 7)
Average content length: 986.3 characters
Total words processed: 30,639,468
Stop words removed: 10,257,229
Stop words removal rate: 33.5%
Categories in dataset: 5
Sourc

In [3]:
df=pd.read_csv("processed_data/cleaned_urdu_news.csv")
# Remove unnecessary columns permanently
columns_to_drop = ['ï»¿Index', 'News Text', 'Source', 'content_before_cleaning']
df.drop(columns=columns_to_drop, inplace=True, errors='ignore')

# Verify the remaining columns
print("Remaining columns:", df.columns.tolist())
print("Final dataset shape:", df.shape)
# Save the final cleaned dataframe
df.to_csv('final_cleaned_urdu_news.csv', index=False, encoding='utf-8')



Remaining columns: ['Headline', 'Category', 'content']
Final dataset shape: (111853, 3)


In [10]:
import os
print(os.getcwd())


/home/cvl/Recommender_System
